In [7]:
import json
import torch
import sys
sys.path.append("..")
from config import TOKENIZER_PATH, MAX_SEQ_LEN
from tokenizer.bpe import ByteLevelBPETokenizer

with open("../data/raw/train.jsonl", "r", encoding="utf-8") as f:
    story = json.loads(next(f))["text"]

tokenizer = ByteLevelBPETokenizer()
tokenizer.load('../'+TOKENIZER_PATH)

token_ids = tokenizer.encode(story)
token_ids = token_ids[:MAX_SEQ_LEN + 1]

input_ids = torch.tensor(token_ids[:-1], dtype=torch.long)
target_ids = torch.tensor(token_ids[1:], dtype=torch.long)

print("Story:", story[:200], "...")
print("Token count:", len(token_ids))
print("Input shape:", input_ids.shape)
print("Target shape:", target_ids.shape)

Story: Once upon a time, in a small house, there lived a clumsy dog named Spot. Spot liked to play and run all day. In the evening, when the sun went down, Spot would get very tired.
One evening, after a lon ...
Token count: 170
Input shape: torch.Size([169])
Target shape: torch.Size([169])


In [8]:
print("Input IDs: ", input_ids[:20])
print("Target IDs:", target_ids[:20])

assert torch.equal(input_ids[1:], target_ids[:-1])

print("\nNext-token prediction setup verified.")

Input IDs:  tensor([ 430,  437,  259,  398,   44,  314,  259,  558,  593,   44,  400,  630,
         259, 2781,  464,  401,  587,   46,  587,  524])
Target IDs: tensor([ 437,  259,  398,   44,  314,  259,  558,  593,   44,  400,  630,  259,
        2781,  464,  401,  587,   46,  587,  524,  266])

Next-token prediction setup verified.


In [9]:
lengths = []

with open("../data/raw/train.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        story = json.loads(line)["text"]
        lengths.append(len(tokenizer.encode(story)))

print("Stories:", len(lengths))
print("Min:", min(lengths))
print("Max:", max(lengths))
print("Mean:", sum(lengths) / len(lengths))

for p in [25, 50, 75, 90, 95, 99]:
    print(f"P{p}:", torch.tensor(lengths).float().quantile(p / 100).item())

Stories: 25000
Min: 63
Max: 1009
Mean: 196.30468
P25: 152.0
P50: 175.0
P75: 206.0
P90: 266.0
P95: 372.0
P99: 573.0


## Dataset Length Analysis

The tokenized TinyStories have variable lengths:

- Median: ~175 tokens
- P90: ~266 tokens
- P99: ~573 tokens
- Maximum: ~1,009 tokens

With `MAX_SEQ_LEN = 512`, padding every story to 512 tokens would waste significant computation because most stories are much shorter.

For efficient training, we will:

1. Tokenize each story.
2. Append an EOS token to preserve story boundaries.
3. Concatenate the tokenized stories into a continuous token stream.
4. Split the stream into fixed-length chunks of `MAX_SEQ_LEN`.
5. Create `input_ids` and `target_ids` using a one-token shift.

This produces efficient fixed-length training batches without unnecessary padding while retaining explicit boundaries between stories.